# 03. 대규모 학습 기법

## 학습 목표
- Mixed Precision Training의 원리와 FP32/FP16/BF16 차이 이해
- PyTorch AMP (Automatic Mixed Precision) 사용법 습득
- Gradient Accumulation으로 작은 GPU에서 큰 배치 시뮬레이션
- Gradient Checkpointing의 메모리 vs 속도 트레이드오프 이해
- Data Parallelism (DDP) 개념 파악

## 참고 자료
- [PyTorch AMP Documentation](https://pytorch.org/docs/stable/amp.html)
- [HuggingFace - Efficient Training](https://huggingface.co/docs/transformers/perf_train_gpu_one)

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time
import struct

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Mixed Precision Training: FP32 vs FP16 vs BF16

### 숫자 표현 형식

| 형식 | 비트 | 범위 | 정밀도 | 용도 |
|------|------|------|--------|------|
| FP32 | 32 | $\pm 3.4 \times 10^{38}$ | 매우 높음 | 기본 학습 |
| FP16 | 16 | $\pm 6.5 \times 10^{4}$ | 낮음 | Mixed Precision |
| BF16 | 16 | $\pm 3.4 \times 10^{38}$ | FP16보다 낮음 | Ampere+ GPU |

### IEEE 754 부동소수점 구조

```
FP32: [1 sign][8 exponent][23 mantissa]  -> 넓은 범위 + 높은 정밀도
FP16: [1 sign][5 exponent][10 mantissa]  -> 좁은 범위 + 낮은 정밀도
BF16: [1 sign][8 exponent][7 mantissa]   -> FP32와 같은 범위 + 낮은 정밀도
```

**BF16의 장점**: FP32와 같은 범위(exponent 8bit)를 가지므로 **overflow/underflow 위험이 적다**.

In [ ]:
# FP32, FP16, BF16 비교

def show_float_info(dtype, name):
    info = torch.finfo(dtype)
    print(f"{name}:")
    print(f"  bits: {info.bits}")
    print(f"  max:  {info.max:.3e}")
    print(f"  min:  {info.min:.3e}")
    print(f"  smallest normal: {info.tiny:.3e}")
    print(f"  eps (precision): {info.eps:.3e}")
    print()

show_float_info(torch.float32, 'FP32')
show_float_info(torch.float16, 'FP16')
show_float_info(torch.bfloat16, 'BF16')

In [ ]:
# Overflow / Precision 문제 시연

print("=== Overflow 문제 (FP16) ===")
val = torch.tensor(60000.0, dtype=torch.float32)
print(f"FP32: {val.item():.1f}")
print(f"FP16: {val.half().item():.1f}")

val2 = torch.tensor(70000.0, dtype=torch.float32)
print(f"\nFP32: {val2.item():.1f}")
print(f"FP16: {val2.half().item()} <- FP16 max = 65504, overflow!")
print(f"BF16: {val2.bfloat16().item():.1f} <- BF16는 범위가 FP32와 같아서 OK")

print(f"\n=== Precision 문제 ===")
a = torch.tensor(1.0, dtype=torch.float32)
b = torch.tensor(0.0001, dtype=torch.float32)
print(f"FP32: 1.0 + 0.0001 = {(a + b).item():.6f}")
print(f"FP16: 1.0 + 0.0001 = {(a.half() + b.half()).item():.6f}")
print(f"BF16: 1.0 + 0.0001 = {(a.bfloat16() + b.bfloat16()).item():.6f}")
print("-> FP16/BF16는 정밀도가 낮아 작은 값이 소실될 수 있음")

print(f"\n=== 메모리 절약 ===")
x_fp32 = torch.randn(1000, 1000, dtype=torch.float32)
x_fp16 = x_fp32.half()
x_bf16 = x_fp32.bfloat16()
print(f"FP32 (1000x1000): {x_fp32.nbytes / 1024 / 1024:.2f} MB")
print(f"FP16 (1000x1000): {x_fp16.nbytes / 1024 / 1024:.2f} MB")
print(f"BF16 (1000x1000): {x_bf16.nbytes / 1024 / 1024:.2f} MB")
print(f"-> 16bit는 32bit 대비 메모리 50% 절약")

In [ ]:
# 실수 형식 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 비트 구조
ax = axes[0]
formats = [
    ('FP32', 1, 8, 23, 'steelblue'),
    ('FP16', 1, 5, 10, 'coral'),
    ('BF16', 1, 8, 7, 'green'),
]

for i, (name, sign, exp, mant, color) in enumerate(formats):
    y = 2 - i
    total = sign + exp + mant
    # Sign
    ax.barh(y, sign, left=0, color='gray', edgecolor='black', height=0.5)
    ax.text(sign / 2, y, f'S({sign})', ha='center', va='center', fontsize=8)
    # Exponent
    ax.barh(y, exp, left=sign, color=color, edgecolor='black', height=0.5, alpha=0.7)
    ax.text(sign + exp / 2, y, f'Exp({exp})', ha='center', va='center', fontsize=8)
    # Mantissa
    ax.barh(y, mant, left=sign + exp, color=color, edgecolor='black', height=0.5, alpha=0.3)
    ax.text(sign + exp + mant / 2, y, f'Mantissa({mant})', ha='center', va='center', fontsize=8)
    # Label
    ax.text(-1, y, f'{name} ({total}bit)', ha='right', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(-6, 34)
ax.set_ylim(-0.5, 3)
ax.set_xlabel('Bits')
ax.set_title('Floating Point Formats: Bit Allocation')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# 오른쪽: 표현 가능 범위
ax = axes[1]
ranges = {
    'FP32': (1.2e-38, 3.4e38),
    'FP16': (6.1e-5, 6.5e4),
    'BF16': (1.2e-38, 3.4e38),
}
colors = ['steelblue', 'coral', 'green']
for i, ((name, (lo, hi)), color) in enumerate(zip(ranges.items(), colors)):
    ax.barh(i, np.log10(hi) - np.log10(lo), left=np.log10(lo),
            color=color, alpha=0.7, edgecolor='black', height=0.5)
    ax.text(0, i, name, ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('log10(value)')
ax.set_title('Representable Range (log scale)')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---
## 2. AMP (Automatic Mixed Precision)

### Mixed Precision의 핵심 아이디어

**모든 연산을 FP16으로 하면 정밀도 문제가 생긴다.** 그래서:

1. **Forward/Backward**: FP16으로 빠르게 계산
2. **Master Weights**: FP32로 유지 (gradient 누적 정밀도 보장)
3. **Loss Scaling**: FP16에서 gradient가 underflow되지 않도록 Loss를 키웠다가 줄임

```
Forward:  FP16 weights -> FP16 activations -> FP16 loss
               |                                  |
               |                        Loss Scaling (x 1024)
               |                                  |
Backward:      |              FP16 gradients <- scaled loss
               |                   |
Update:   FP32 master weights <- FP32 gradients (unscaled)
               |
          FP32 -> FP16 copy for next forward
```

### PyTorch AMP: `autocast` + `GradScaler`
- `autocast`: 연산별로 적절한 dtype 자동 선택
- `GradScaler`: Loss Scaling 자동 관리

In [ ]:
# 간단한 MLP 모델 정의

class SimpleMLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, output_dim=10, num_layers=4):
        super().__init__()
        layers = []
        # 첫 레이어
        layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU()])
        # 중간 레이어
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), nn.ReLU()])
        # 출력 레이어
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = SimpleMLP()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params:,} parameters")
print(f"FP32 메모리: {total_params * 4 / 1024 / 1024:.2f} MB")
print(f"FP16 메모리: {total_params * 2 / 1024 / 1024:.2f} MB")

In [ ]:
# AMP 학습 vs 일반 학습 비교
from torch.amp import autocast, GradScaler

def train_step_fp32(model, data, target, optimizer, criterion):
    """FP32 학습 (\uae30\ubcf8)"""
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    return loss.item()


def train_step_amp(model, data, target, optimizer, criterion, scaler):
    """AMP \ud559\uc2b5 (Mixed Precision)"""
    optimizer.zero_grad()

    # autocast: forward pass\ub97c FP16\uc73c\ub85c \uc790\ub3d9 \uc218\ud589
    with autocast(device_type=device.type):
        output = model(data)
        loss = criterion(output, target)

    # GradScaler: loss\ub97c \ud0a4\uc6e0\ub2e4\uac00 gradient\ub97c \uc904\uc784 (FP16 underflow \ubc29\uc9c0)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    return loss.item()


# 학습 코드 비교 출력
print("=== FP32 Training ===")
print("""
optimizer.zero_grad()
output = model(data)         # FP32
loss = criterion(output, target)
loss.backward()              # FP32 gradients
optimizer.step()             # FP32 update
""")

print("=== AMP Training ===")
print("""
optimizer.zero_grad()
with autocast(device_type='cuda'):  # <- FP16 forward
    output = model(data)
    loss = criterion(output, target)
scaler.scale(loss).backward()  # <- scaled FP16 gradients
scaler.step(optimizer)         # <- unscale + FP32 update
scaler.update()                # <- adjust scale factor
""")

In [ ]:
# autocast 동작 시연 (CPU에서도 동작)
print("autocast 내부에서 dtype 변화 확인:")
print()

linear = nn.Linear(10, 10)
x = torch.randn(2, 10)

# 일반 연산
out_normal = linear(x)
print(f"Without autocast:")
print(f"  input dtype:  {x.dtype}")
print(f"  weight dtype: {linear.weight.dtype}")
print(f"  output dtype: {out_normal.dtype}")

# autocast 사용
with autocast(device_type='cpu', dtype=torch.float16):
    out_amp = linear(x)
    print(f"\nWith autocast:")
    print(f"  input dtype:  {x.dtype}")
    print(f"  weight dtype: {linear.weight.dtype}  (master weight는 FP32 유지)")
    print(f"  output dtype: {out_amp.dtype}  (<- 자동으로 FP16!)")

print(f"\n-> autocast는 weight를 복사해서 FP16으로 변환, 원본 weight는 FP32 유지")

---
## 3. Gradient Accumulation: 큰 배치를 작은 GPU에서 시뮬레이션

### 문제
- LLM 학습에는 큰 배치 사이즈가 필요 (GPT-3: batch size 3.2M tokens)
- 하지만 GPU 메모리에 큰 배치가 들어가지 않음

### 해결책
- 작은 mini-batch로 여러 번 forward/backward를 수행
- gradient를 **누적**한 후 한 번에 업데이트

$$\nabla L_{\text{effective}} = \frac{1}{K} \sum_{k=1}^{K} \nabla L_k$$

- $K$ = accumulation steps
- 실효 배치 사이즈 = mini-batch size $\times K$

```
Gradient Accumulation (K=4, mini-batch=8):

Step 1: forward(batch_1) -> backward() -> grad += grad_1
Step 2: forward(batch_2) -> backward() -> grad += grad_2
Step 3: forward(batch_3) -> backward() -> grad += grad_3
Step 4: forward(batch_4) -> backward() -> grad += grad_4
                                          grad /= 4
                                          optimizer.step()  -> 실효 batch=32
                                          optimizer.zero_grad()
```

In [ ]:
# Gradient Accumulation 구현

def train_with_grad_accumulation(model, data_batches, optimizer, criterion,
                                  accumulation_steps=4):
    """
    Gradient Accumulation \ud559\uc2b5 \ub8e8\ud504
    accumulation_steps\ubc88\ub9c8\ub2e4 1\ubc88 weight update
    """
    total_loss = 0
    optimizer.zero_grad()

    for i, (data, target) in enumerate(data_batches):
        # Forward
        output = model(data)
        loss = criterion(output, target)

        # Normalize loss by accumulation steps
        loss = loss / accumulation_steps
        loss.backward()  # gradient가 누적됨 (zero_grad를 안 했으니까)

        total_loss += loss.item()

        # K step마다 weight update
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            print(f"  Step {i+1}: weight update! (accumulated {accumulation_steps} mini-batches)")

    return total_loss


# 시연
model = SimpleMLP(input_dim=64, hidden_dim=128, output_dim=10, num_layers=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 8개 mini-batch, 각 batch_size=16
mini_batch_size = 16
accumulation_steps = 4
effective_batch = mini_batch_size * accumulation_steps

data_batches = [(torch.randn(mini_batch_size, 64),
                 torch.randint(0, 10, (mini_batch_size,)))
                for _ in range(8)]

print(f"Mini-batch size: {mini_batch_size}")
print(f"Accumulation steps: {accumulation_steps}")
print(f"Effective batch size: {effective_batch}")
print(f"Total mini-batches: {len(data_batches)}")
print(f"Weight updates: {len(data_batches) // accumulation_steps}")
print()

loss = train_with_grad_accumulation(model, data_batches, optimizer, criterion,
                                     accumulation_steps=accumulation_steps)

In [ ]:
# Gradient Accumulation이 정말로 큰 배치와 같은 결과를 내는지 검증
torch.manual_seed(42)

# 같은 데이터 준비
full_data = torch.randn(32, 64)
full_target = torch.randint(0, 10, (32,))

# 방법 1: 큰 배치 한 번에
torch.manual_seed(0)
model1 = SimpleMLP(input_dim=64, hidden_dim=32, output_dim=10, num_layers=2)
opt1 = torch.optim.SGD(model1.parameters(), lr=0.01)

opt1.zero_grad()
loss1 = criterion(model1(full_data), full_target)
loss1.backward()
grad1 = model1.net[0].weight.grad.clone()

# 방법 2: Gradient Accumulation (4 x 8)
torch.manual_seed(0)
model2 = SimpleMLP(input_dim=64, hidden_dim=32, output_dim=10, num_layers=2)
opt2 = torch.optim.SGD(model2.parameters(), lr=0.01)

opt2.zero_grad()
for k in range(4):
    mini_data = full_data[k*8:(k+1)*8]
    mini_target = full_target[k*8:(k+1)*8]
    loss2 = criterion(model2(mini_data), mini_target) / 4  # 누적이니까 /4
    loss2.backward()

grad2 = model2.net[0].weight.grad.clone()

# 비교
print("Gradient 비교 (\uccab \ub808\uc774\uc5b4 weight):")
print(f"  Big batch grad (norm):     {grad1.norm().item():.6f}")
print(f"  Accumulated grad (norm):   {grad2.norm().item():.6f}")
print(f"  Difference:                {(grad1 - grad2).norm().item():.8f}")
print(f"  Nearly equal: {torch.allclose(grad1, grad2, atol=1e-5)}")
print()
print("-> 예\uc0c1: CrossEntropyLoss\ub294 \uae30\ubcf8\uc801\uc73c\ub85c mean\uc744 \uc0ac\uc6a9\ud558\ubbc0\ub85c")
print("   full batch\uc758 mean\uacfc accumulated mean\uc774 \uc815\ud655\ud788 \uac19\uc9c0\ub294 \uc54a\uc744 \uc218 \uc788\uc74c")
print("   \ud558\uc9c0\ub9cc \ucda9\ubd84\ud788 \uc720\uc0ac\ud558\uac8c \ub3d9\uc791")

---
## 4. Gradient Checkpointing: 메모리 절약, 속도 트레이드오프

### 문제
- Backward pass에서 gradient를 계산하려면 **forward의 중간 결과(activation)**가 필요
- Transformer는 레이어가 많아서 activation 메모리가 엄청남

### 해결책: Gradient Checkpointing
- 일부 레이어의 activation만 저장 (checkpoint)
- 나머지는 backward 시 다시 계산 (recomputation)
- **메모리**: $O(\sqrt{L})$로 감소 (L = 레이어 수)
- **속도**: ~30% 느려짐 (recomputation 비용)

```
Normal:       [L1] -> act1 -> [L2] -> act2 -> [L3] -> act3 -> [L4]
                       |               |               |
              메모리에 저장    메모리에 저장    메모리에 저장

Checkpointing: [L1] -> act1 -> [L2] -> (X) -> [L3] -> act3 -> [L4]
                       |                               |
                  checkpoint                       checkpoint
                  (backward시 act2는 act1에서 다시 계산)
```

In [ ]:
# Gradient Checkpointing 메모리 절약 시뮬레이션
from torch.utils.checkpoint import checkpoint

class ModelWithCheckpointing(nn.Module):
    def __init__(self, hidden_dim=256, num_layers=8):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
            for _ in range(num_layers)
        ])
        self.head = nn.Linear(hidden_dim, 10)
        self.use_checkpointing = False

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpointing:
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return self.head(x)


# 메모리 사용량 측정 (텐서 사이즈로 추정)
def estimate_activation_memory(model, input_data, use_checkpoint=False):
    """Activation 메모리를 추\uc815"""
    model.use_checkpointing = use_checkpoint

    # Forward pass를 수\ud589\ud558\uba74\uc11c activation 수\ub97c \uce21\uc815
    activations = []
    hooks = []

    def hook_fn(module, input, output):
        if isinstance(output, torch.Tensor):
            activations.append(output.nelement() * output.element_size())

    if not use_checkpoint:
        for layer in model.layers:
            hooks.append(layer.register_forward_hook(hook_fn))

    output = model(input_data)
    loss = output.sum()
    loss.backward()

    for h in hooks:
        h.remove()

    total_bytes = sum(activations)
    return total_bytes


# 바이트 수로 직접 계산
hidden_dim = 256
batch_size = 64
num_layers = 8

# Normal: 모든 레이어의 activation 저장
act_per_layer = batch_size * hidden_dim * 4  # FP32 = 4 bytes
normal_memory = act_per_layer * num_layers

# Checkpointing: sqrt(L)개 레이어만 저장
checkpoint_layers = int(np.ceil(np.sqrt(num_layers)))
checkpoint_memory = act_per_layer * checkpoint_layers

print(f"\ubaa8\ub378: {num_layers} layers, hidden_dim={hidden_dim}, batch_size={batch_size}")
print(f"\nActivation \uba54\ubaa8\ub9ac (\ucd94\uc815):")
print(f"  Normal:        {normal_memory / 1024:.1f} KB ({num_layers} layers \uc800\uc7a5)")
print(f"  Checkpointing: {checkpoint_memory / 1024:.1f} KB ({checkpoint_layers} layers \uc800\uc7a5)")
print(f"  \uc808\uc57d:         {(1 - checkpoint_memory / normal_memory) * 100:.0f}%")
print(f"\n\uc2e4\uc81c LLM (96 layers): ")
print(f"  Normal:        96 layers \uc800\uc7a5")
print(f"  Checkpointing: ~{int(np.sqrt(96))} layers \uc800\uc7a5 -> {(1 - np.sqrt(96)/96) * 100:.0f}% \uc808\uc57d")

---
## 5. Data Parallelism (DDP): 여러 GPU로 학습

### 개념
- 모델을 모든 GPU에 복사
- 데이터를 GPU별로 나누어 처리
- Gradient를 모든 GPU에서 평균 (All-Reduce)

```
Data Parallelism (4 GPUs, batch=128):

GPU 0: model copy + batch[0:32]   -> grad_0
GPU 1: model copy + batch[32:64]  -> grad_1   --> All-Reduce --> avg_grad
GPU 2: model copy + batch[64:96]  -> grad_2                       |
GPU 3: model copy + batch[96:128] -> grad_3                  all GPUs update
                                                             with avg_grad
```

### DDP vs DataParallel

| | `nn.DataParallel` | `DistributedDataParallel` |
|---|---|---|
| 통신 | GPU 0이 모두 담당 | All-Reduce (분산) |
| 성능 | GPU 0 병목 | 균등한 부하 |
| 추천 | 비추천 | **항상 이것 사용** |

In [ ]:
# DDP 시각화: All-Reduce 개념
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Step 1: 각 GPU에서 독립적으로 forward/backward
ax = axes[0]
gpu_labels = ['GPU 0', 'GPU 1', 'GPU 2', 'GPU 3']
grad_values = [0.3, -0.1, 0.5, 0.1]  # 각 GPU의 gradient
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']

bars = ax.bar(gpu_labels, grad_values, color=colors, edgecolor='black')
ax.set_ylabel('Gradient Value')
ax.set_title('Step 1: Independent Forward/Backward')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)

for bar, val in zip(bars, grad_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10)

# Step 2: All-Reduce
ax = axes[1]
avg_grad = np.mean(grad_values)
ax.bar(gpu_labels, [avg_grad] * 4, color='lightgreen', edgecolor='black')
ax.set_ylabel('Gradient Value')
ax.set_title(f'Step 2: All-Reduce (avg = {avg_grad:.2f})')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.set_ylim(min(grad_values) - 0.1, max(grad_values) + 0.1)

for i in range(4):
    ax.text(i, avg_grad + 0.02, f'{avg_grad:.2f}', ha='center', va='bottom', fontsize=10)

# Step 3: Scaling 효율
ax = axes[2]
gpus = [1, 2, 4, 8, 16, 32, 64]
ideal_speedup = gpus
# 실제는 통신 오버헤드로 선형보다 느림
real_speedup = [1, 1.9, 3.7, 7.0, 13.0, 23.0, 38.0]

ax.plot(gpus, ideal_speedup, 'k--', label='Ideal (Linear)', linewidth=1.5)
ax.plot(gpus, real_speedup, 'b-o', label='Typical DDP', linewidth=2)
ax.set_xlabel('Number of GPUs')
ax.set_ylabel('Speedup')
ax.set_title('DDP Scaling Efficiency')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("통\uc2e0 \uc624\ubc84\ud5e4\ub4dc\ub85c GPU \uc218\uac00 \ub9ce\uc744\uc218\ub85d \ud6a8\uc728 \uc800\ud558")
print("\ud558\uc9c0\ub9cc DDP\ub294 \uac00\uc7a5 \uc27d\uace0 \ud6a8\uc728\uc801\uc778 \uba40\ud2f0 GPU \ud559\uc2b5 \ubc29\ubc95")

In [ ]:
# DDP 코드 구조 (예시 - 실행은 multi-GPU 필요)
print("DDP 학습 코드 구조:")
print("""
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

# 1. 분산 환경 초기화
dist.init_process_group(backend='nccl')  # NVIDIA GPU는 nccl
local_rank = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local_rank)

# 2. 모델을 DDP로 감싸기
model = MyModel().to(local_rank)
model = DDP(model, device_ids=[local_rank])

# 3. DistributedSampler로 데이터 분배
sampler = DistributedSampler(dataset)
dataloader = DataLoader(dataset, sampler=sampler)

# 4. 학습 루프는 동일
for data, target in dataloader:
    loss = criterion(model(data), target)
    loss.backward()    # DDP가 자동으로 All-Reduce
    optimizer.step()
    optimizer.zero_grad()

# 실행: torchrun --nproc_per_node=4 train.py
""")

---
## 6. 소규모 실험: MLP에 AMP + Gradient Accumulation 적용

위에서 배운 기법들을 하나의 학습 루프에 함께 적용해보자.

In [ ]:
# 종합 실험: 다양한 설정으로 학습 시간/메모리 비교

def train_experiment(use_amp=False, accumulation_steps=1,
                     hidden_dim=512, num_layers=4,
                     batch_size=128, num_steps=100):
    """다\uc591\ud55c \uc124\uc815\uc73c\ub85c \ud559\uc2b5 \uc2e4\ud5d8"""
    model = SimpleMLP(input_dim=784, hidden_dim=hidden_dim,
                      output_dim=10, num_layers=num_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() if use_amp else None

    mini_batch = batch_size // accumulation_steps
    losses = []

    start_time = time.time()

    for step in range(num_steps):
        optimizer.zero_grad()
        step_loss = 0

        for acc_step in range(accumulation_steps):
            data = torch.randn(mini_batch, 784).to(device)
            target = torch.randint(0, 10, (mini_batch,)).to(device)

            if use_amp:
                with autocast(device_type=device.type):
                    output = model(data)
                    loss = criterion(output, target) / accumulation_steps
                scaler.scale(loss).backward()
            else:
                output = model(data)
                loss = criterion(output, target) / accumulation_steps
                loss.backward()

            step_loss += loss.item()

        if use_amp:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()

        losses.append(step_loss)

    elapsed = time.time() - start_time

    # 모델 메모리
    param_memory = sum(p.nbytes for p in model.parameters()) / 1024 / 1024

    return {
        'losses': losses,
        'time': elapsed,
        'param_memory_mb': param_memory,
        'final_loss': np.mean(losses[-10:]),
    }


# 실험 수행
configs = [
    {'name': 'FP32 (baseline)',       'use_amp': False, 'accumulation_steps': 1},
    {'name': 'AMP (FP16)',            'use_amp': True,  'accumulation_steps': 1},
    {'name': 'FP32 + GradAccum(4)',   'use_amp': False, 'accumulation_steps': 4},
    {'name': 'AMP + GradAccum(4)',    'use_amp': True,  'accumulation_steps': 4},
]

results = {}
for cfg in configs:
    name = cfg.pop('name')
    print(f"Running: {name}...", end=" ")
    result = train_experiment(**cfg, num_steps=50)
    results[name] = result
    print(f"time={result['time']:.2f}s, loss={result['final_loss']:.4f}")

print(f"\nDevice: {device}")
print("\uc8fc\uc758: CPU\uc5d0\uc11c\ub294 AMP \ud6a8\uacfc\uac00 \uc81c\ud55c\uc801. GPU\uc5d0\uc11c \uc2e4\ud589\ud558\uba74 \uc0c1\ub2f9\ud55c \uc18d\ub3c4 \ucc28\uc774 \ub0a0 \uc218 \uc788\uc74c.")

In [ ]:
# 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: Loss curves
ax = axes[0]
for name, result in results.items():
    ax.plot(result['losses'], label=name, linewidth=1.5, alpha=0.8)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 오른쪽: 시간 비교
ax = axes[1]
names = list(results.keys())
times = [results[n]['time'] for n in names]
colors = ['steelblue', 'coral', 'green', 'purple']

bars = ax.bar(range(len(names)), times, color=colors, edgecolor='black', alpha=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Time (seconds)')
ax.set_title('Training Time Comparison')
ax.grid(True, alpha=0.3, axis='y')

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{t:.2f}s', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: AMP + Gradient Accumulation 종합 학습 루프

아래 조건으로 종합 학습 루프를 구현하세요:
- MLP 모델 (input_dim=256, hidden_dim=512, output_dim=10, num_layers=6)
- AMP 사용 (autocast + GradScaler)
- Gradient Accumulation 8 steps (mini-batch=32, 실효 batch=256)
- 100 epoch 학습 후 loss curve 시각화
- AMP 없이 같은 조건으로 학습한 결과와 시간/loss 비교

In [ ]:
# TODO: 종합 학습 루프 구현
# 1. SimpleMLP 모델 생성 (input_dim=256, hidden_dim=512, output_dim=10, num_layers=6)
# 2. AMP 학습 루프:
#    - mini_batch = 32
#    - accumulation_steps = 8
#    - autocast + GradScaler 사용
#    - 100 step 학습
# 3. FP32 학습 루프 (\uac19\uc740 \uc870\uac74, AMP\ub9cc \uc81c\uc678)
# 4. \ub450 \uacb0\uacfc\uc758 loss curve\uc640 \uc2dc\uac04\uc744 \ube44\uad50 \uc2dc\uac01\ud654


---
## 핵심 정리

| 기법 | 효과 | 트레이드오프 | 사용 시기 |
|------|------|----------|----------|
| Mixed Precision (AMP) | 속도 ~2x, 메모리 ~50% 절약 | 약간의 정밀도 손실 | 항상 사용 |
| Gradient Accumulation | 큰 실효 배치 사용 | 속도 변화 없음 | GPU 메모리 부족 시 |
| Gradient Checkpointing | 메모리 $O(\sqrt{L})$ | 속도 ~30% 감소 | 메모리 부족 시 |
| DDP | 선형에 가까운 speedup | 통신 오버헤드 | Multi-GPU 학습 시 |
| BF16 vs FP16 | BF16이 더 안정적 | BF16 지원 GPU 필요 | Ampere+ GPU |

**다음 노트북**: [04-data-pipeline.ipynb](04-data-pipeline.ipynb) - 데이터 파이프라인